## schema survey

In [1]:
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict
import random

import re
DEBATES_DIR = Path("data/debates")

def sample_files(debates_dir, per_decade=5):
    all_files = sorted(debates_dir.glob("debates*.xml"))
    by_decade = defaultdict(list)
    for f in all_files:
        m = re.search(r'(\d{4})', f.stem)
        if m:
            year = int(m.group(1))
            by_decade[(year // 10) * 10].append(f)
    sampled = []
    for decade in sorted(by_decade):
        files = by_decade[decade]
        sampled.extend(random.sample(files, min(per_decade, len(files))))
    return sampled

def survey_file(path):
    m = re.search(r'(\d{4})', path.stem)
    year = m.group(1) if m else "????"
    try:
        tree = ET.parse(path)
        root = tree.getroot()
    except ET.ParseError as e:
        return {"file": path.name, "error": str(e)}

    tag_attrs = defaultdict(set)
    for el in root.iter():
        tag_attrs[el.tag].update(el.attrib.keys())

    speech_tags = {t: attrs for t, attrs in tag_attrs.items()
                   if any(k in t.lower() for k in ("speech", "question", "answer", "oral", "written"))}
    heading_tags = {t: attrs for t, attrs in tag_attrs.items()
                    if any(k in t.lower() for k in ("head", "title", "section"))}

    speech_sample = {}
    for tag in speech_tags:
        el = root.find(f".//{tag}")
        if el is not None:
            speech_sample[tag] = dict(el.attrib)
            break

    return {
        "file":          path.name,
        "year":          year,
        "root_tag":      root.tag,
        "all_tags":      sorted(tag_attrs.keys()),
        "speech_tags":   {t: sorted(a) for t, a in speech_tags.items()},
        "heading_tags":  {t: sorted(a) for t, a in heading_tags.items()},
        "speech_sample": speech_sample,
    }

files = sample_files(DEBATES_DIR)
print(f"Surveying {len(files)} files...\n")

results = []
for f in files:
    r = survey_file(f)
    results.append(r)
    if "error" in r:
        print(f"  ERROR {r['file']}: {r['error']}")
    else:
        print(f"  {r['year']}  {r['file']:<55}  root=<{r['root_tag']}>  speech_tags={list(r['speech_tags'].keys())}")

print("\n--- Speech tag breakdown by year ---")
tag_years = defaultdict(list)
for r in results:
    if "error" not in r:
        for tag in r["speech_tags"]:
            tag_years[tag].append(r["year"])
for tag, years in sorted(tag_years.items()):
    print(f"  <{tag}>: {sorted(years)}")

print("\n--- Speech element attributes by year ---")
for r in results:
    if "error" not in r and r["speech_sample"]:
        for tag, attrs in r["speech_sample"].items():
            print(f"  {r['year']}  <{tag}> attrs: {attrs}")

print("\n--- All unique tags seen ---")
all_tags = sorted({t for r in results if "error" not in r for t in r["all_tags"]})
print(" ", ", ".join(all_tags))

Surveying 60 files...

  1919  debates1919-07-08a.xml                                   root=<publicwhip>  speech_tags=['speech']
  1919  debates1919-11-24a.xml                                   root=<publicwhip>  speech_tags=['speech']
  1919  debates1919-07-01a.xml                                   root=<publicwhip>  speech_tags=['speech']
  1919  debates1919-06-30a.xml                                   root=<publicwhip>  speech_tags=['speech']
  1919  debates1919-08-07a.xml                                   root=<publicwhip>  speech_tags=['speech']
  1922  debates1922-05-04a.xml                                   root=<publicwhip>  speech_tags=['speech']
  1925  debates1925-03-16a.xml                                   root=<publicwhip>  speech_tags=['speech']
  1923  debates1923-07-12a.xml                                   root=<publicwhip>  speech_tags=['speech']
  1927  debates1927-07-07a.xml                                   root=<publicwhip>  speech_tags=['speech']
  1926  debate

In [3]:
import numpy as np

counts = [by_year[y] for y in years]

print(f"Total files:  {sum(counts)}")
print(f"Year range:   {years[0]}–{years[-1]}")
print(f"Years with data: {len(years)}")
print(f"\nFiles per year:")
print(f"  Mean:    {np.mean(counts):.1f}")
print(f"  Median:  {np.median(counts):.1f}")
print(f"  Std dev: {np.std(counts):.1f}")
print(f"  Min:     {min(counts)}  ({years[counts.index(min(counts))]})")
print(f"  Max:     {max(counts)}  ({years[counts.index(max(counts))]})")

print(f"\nBottom 5 years:")
for y, c in sorted(by_year.items(), key=lambda x: x[1])[:5]:
    print(f"  {y}: {c} files")

print(f"\nTop 5 years:")
for y, c in sorted(by_year.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {y}: {c} files")

# Any gaps
all_years = set(range(years[0], years[-1]+1))
gaps = sorted(all_years - set(years))
if gaps:
    print(f"\nYears with no files: {gaps}")
else:
    print(f"\nNo gaps — every year has at least one file")

Total files:  19977
Year range:   1919–2026
Years with data: 108

Files per year:
  Mean:    185.0
  Median:  161.0
  Std dev: 79.2
  Min:     62  (1923)
  Max:     473  (2025)

Bottom 5 years:
  1923: 62 files
  1921: 64 files
  1942: 98 files
  1924: 102 files
  1929: 102 files

Top 5 years:
  2025: 473 files
  2024: 443 files
  2020: 410 files
  2009: 373 files
  2008: 372 files

No gaps — every year has at least one file
